In [ ]:
import os
import time

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from lerobot.datasets.lerobot_dataset import LeRobotDataset


# ============================================================
# 1. CONFIG
# ============================================================

DATASET_ID = "lerobot/aloha_sim_transfer_cube_human"

SEED = 42
TRAIN_RATIO = 0.8

CHUNK_SIZE = 5

BATCH_SIZE = 128
EPOCHS = 3
LR = 1e-3

# Keep small so it runs fast.
MAX_TRAIN = 3000
MAX_VAL = 800

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs("results", exist_ok=True)

print("Device:", DEVICE)


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def to_numpy(x):
    """Convert tensor to numpy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)


def get_chunk_indices(episode_list, starts, ends, chunk_size):
    """
    Create valid starting indices for action chunks.

    If CHUNK_SIZE = 5, then each example needs:
    action[t], action[t+1], action[t+2], action[t+3], action[t+4]

    So we cannot start too close to the end of an episode.
    """
    indices = []

    for ep in episode_list:
        start = starts[ep]
        end = ends[ep]

        last_valid_start = end - chunk_size

        for idx in range(start, last_valid_start + 1):
            indices.append(idx)

    return np.array(indices)


def shuffle_and_limit(indices, max_count):
    """
    Shuffle examples and keep a smaller subset.
    """
    np.random.shuffle(indices)

    if len(indices) > max_count:
        indices = indices[:max_count]

    return indices


def smoothness_score(action_sequence):
    """
    Measures how much actions change from step to step.

    Lower = smoother.
    Higher = jerkier.

    action_sequence shape:
        [timesteps, action_dim]
    """
    diffs = np.diff(action_sequence, axis=0)
    return np.mean(np.linalg.norm(diffs, axis=1))


# ============================================================
# 3. LOAD DATASET
# ============================================================

print("\nLoading dataset...")

dataset = LeRobotDataset(DATASET_ID)
sample = dataset[0]

state_dim = to_numpy(sample["observation.state"]).shape[0]
action_dim = to_numpy(sample["action"]).shape[0]

print("Dataset loaded.")
print("Total timesteps:", len(dataset))
print("State dim:", state_dim)
print("Action dim:", action_dim)


# ============================================================
# 4. EPISODE-BASED TRAIN / VAL SPLIT
# ============================================================

episode_info = dataset.episode_data_index

starts = to_numpy(episode_info["from"]).astype(int)
ends = to_numpy(episode_info["to"]).astype(int)

num_episodes = len(starts)

episodes = np.arange(num_episodes)
np.random.shuffle(episodes)

split_point = int(TRAIN_RATIO * num_episodes)

train_episodes = episodes[:split_point]
val_episodes = episodes[split_point:]

train_indices = get_chunk_indices(
    train_episodes,
    starts,
    ends,
    CHUNK_SIZE
)

val_indices = get_chunk_indices(
    val_episodes,
    starts,
    ends,
    CHUNK_SIZE
)

train_indices = shuffle_and_limit(train_indices, MAX_TRAIN)
val_indices = shuffle_and_limit(val_indices, MAX_VAL)

print("\nEpisode split complete.")
print("Total episodes:", num_episodes)
print("Train episodes:", len(train_episodes))
print("Val episodes:", len(val_episodes))
print("Train chunk examples:", len(train_indices))
print("Val chunk examples:", len(val_indices))


# ============================================================
# 5. ACTION NORMALIZATION
# ============================================================

print("\nComputing action normalization...")

train_actions = []

for idx in train_indices:
    for offset in range(CHUNK_SIZE):
        item = dataset[int(idx + offset)]
        action = to_numpy(item["action"]).astype(np.float32)
        train_actions.append(action)

train_actions = np.stack(train_actions)

action_mean = train_actions.mean(axis=0)
action_std = train_actions.std(axis=0)

# Avoid division by zero.
action_std = np.maximum(action_std, 1e-6)

print("Action mean shape:", action_mean.shape)
print("Action std shape:", action_std.shape)


# ============================================================
# 6. PYTORCH DATASET
# ============================================================

class ActionChunkDataset(Dataset):
    """
    Each example returns:

    state[t]
    action_chunk[t : t+K]
    """

    def __init__(self, dataset, indices, action_mean, action_std, chunk_size):
        self.dataset = dataset
        self.indices = indices
        self.chunk_size = chunk_size

        self.action_mean = torch.tensor(action_mean, dtype=torch.float32)
        self.action_std = torch.tensor(action_std, dtype=torch.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])

        item = self.dataset[idx]

        state = torch.tensor(
            to_numpy(item["observation.state"]),
            dtype=torch.float32
        )

        action_chunk = []

        for offset in range(self.chunk_size):
            future_item = self.dataset[idx + offset]

            action = torch.tensor(
                to_numpy(future_item["action"]),
                dtype=torch.float32
            )

            action_chunk.append(action)

        action_chunk_raw = torch.stack(action_chunk, dim=0)

        action_chunk_norm = (
            action_chunk_raw - self.action_mean
        ) / self.action_std

        return {
            "state": state,
            "action_chunk_norm": action_chunk_norm,
            "action_chunk_raw": action_chunk_raw,
        }


train_ds = ActionChunkDataset(
    dataset=dataset,
    indices=train_indices,
    action_mean=action_mean,
    action_std=action_std,
    chunk_size=CHUNK_SIZE
)

val_ds = ActionChunkDataset(
    dataset=dataset,
    indices=val_indices,
    action_mean=action_mean,
    action_std=action_std,
    chunk_size=CHUNK_SIZE
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))

print("\nBatch check:")
print("State:", batch["state"].shape)
print("Action chunk:", batch["action_chunk_norm"].shape)


# ============================================================
# 7. MODEL 1: ONE-STEP POLICY
# ============================================================

class OneStepPolicy(nn.Module):
    """
    robot_state[t] -> action[t]
    """

    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim),
        )

    def forward(self, state):
        return self.model(state)


# ============================================================
# 8. MODEL 2: ACTION-CHUNK POLICY
# ============================================================

class ActionChunkPolicy(nn.Module):
    """
    robot_state[t] -> actions[t : t+K]
    """

    def __init__(self, state_dim, action_dim, chunk_size):
        super().__init__()

        self.action_dim = action_dim
        self.chunk_size = chunk_size

        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, chunk_size * action_dim),
        )

    def forward(self, state):
        x = self.model(state)

        # Convert:
        # [batch, chunk_size * action_dim]
        # into:
        # [batch, chunk_size, action_dim]
        x = x.view(-1, self.chunk_size, self.action_dim)

        return x


# ============================================================
# 9. TRAIN ONE-STEP MODEL
# ============================================================

def train_one_step(model):
    model = model.to(DEVICE)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\n" + "=" * 60)
    print("Training one-step policy")
    print("=" * 60)

    for epoch in range(1, EPOCHS + 1):
        model.train()

        train_loss_total = 0
        train_count = 0

        for batch in train_loader:
            state = batch["state"].to(DEVICE)

            # One-step target is only the first action in the chunk.
            target = batch["action_chunk_norm"][:, 0, :].to(DEVICE)

            pred = model(state)

            loss = loss_fn(pred, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * state.size(0)
            train_count += state.size(0)

        train_loss = train_loss_total / train_count

        val_loss, raw_mae = evaluate_one_step(model, loss_fn)

        print(
            f"Epoch {epoch} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Raw Val MAE: {raw_mae:.4f}"
        )

    return model, val_loss, raw_mae


def evaluate_one_step(model, loss_fn):
    model.eval()

    val_loss_total = 0
    val_count = 0

    all_preds_raw = []
    all_true_raw = []

    mean = torch.tensor(action_mean, dtype=torch.float32).to(DEVICE)
    std = torch.tensor(action_std, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        for batch in val_loader:
            state = batch["state"].to(DEVICE)

            target_norm = batch["action_chunk_norm"][:, 0, :].to(DEVICE)
            target_raw = batch["action_chunk_raw"][:, 0, :].to(DEVICE)

            pred_norm = model(state)

            loss = loss_fn(pred_norm, target_norm)

            pred_raw = pred_norm * std + mean

            val_loss_total += loss.item() * state.size(0)
            val_count += state.size(0)

            all_preds_raw.append(pred_raw.cpu())
            all_true_raw.append(target_raw.cpu())

    preds = torch.cat(all_preds_raw).numpy()
    true = torch.cat(all_true_raw).numpy()

    raw_mae = np.mean(np.abs(preds - true))
    val_loss = val_loss_total / val_count

    return val_loss, raw_mae


# ============================================================
# 10. TRAIN ACTION-CHUNK MODEL
# ============================================================

def train_chunk(model):
    model = model.to(DEVICE)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\n" + "=" * 60)
    print("Training action-chunk policy")
    print("=" * 60)

    for epoch in range(1, EPOCHS + 1):
        model.train()

        train_loss_total = 0
        train_count = 0

        for batch in train_loader:
            state = batch["state"].to(DEVICE)
            target = batch["action_chunk_norm"].to(DEVICE)

            pred = model(state)

            loss = loss_fn(pred, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_total += loss.item() * state.size(0)
            train_count += state.size(0)

        train_loss = train_loss_total / train_count

        val_loss, raw_mae, per_step_mae = evaluate_chunk(model, loss_fn)

        print(
            f"Epoch {epoch} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Raw Val MAE: {raw_mae:.4f}"
        )

    return model, val_loss, raw_mae, per_step_mae


def evaluate_chunk(model, loss_fn):
    model.eval()

    val_loss_total = 0
    val_count = 0

    all_preds_raw = []
    all_true_raw = []

    mean = torch.tensor(action_mean, dtype=torch.float32).to(DEVICE)
    std = torch.tensor(action_std, dtype=torch.float32).to(DEVICE)

    # Reshape for broadcasting across chunk dimension.
    mean = mean.view(1, 1, -1)
    std = std.view(1, 1, -1)

    with torch.no_grad():
        for batch in val_loader:
            state = batch["state"].to(DEVICE)

            target_norm = batch["action_chunk_norm"].to(DEVICE)
            target_raw = batch["action_chunk_raw"].to(DEVICE)

            pred_norm = model(state)

            loss = loss_fn(pred_norm, target_norm)

            pred_raw = pred_norm * std + mean

            val_loss_total += loss.item() * state.size(0)
            val_count += state.size(0)

            all_preds_raw.append(pred_raw.cpu())
            all_true_raw.append(target_raw.cpu())

    preds = torch.cat(all_preds_raw).numpy()
    true = torch.cat(all_true_raw).numpy()

    raw_mae = np.mean(np.abs(preds - true))

    # Error for each future step inside the chunk.
    # Shape:
    # preds = [examples, chunk_size, action_dim]
    per_step_mae = np.mean(np.abs(preds - true), axis=(0, 2))

    val_loss = val_loss_total / val_count

    return val_loss, raw_mae, per_step_mae


# ============================================================
# 11. SIMPLE INFERENCE TIMING
# ============================================================

def measure_time_one_step(model):
    model.eval()

    sample_state = batch["state"][:1].to(DEVICE)

    start = time.perf_counter()

    with torch.no_grad():
        for _ in range(200):
            _ = model(sample_state)

    end = time.perf_counter()

    return (end - start) / 200


def measure_time_chunk(model):
    model.eval()

    sample_state = batch["state"][:1].to(DEVICE)

    start = time.perf_counter()

    with torch.no_grad():
        for _ in range(200):
            _ = model(sample_state)

    end = time.perf_counter()

    return (end - start) / 200


# ============================================================
# 12. RUN EXPERIMENTS
# ============================================================

results = []

one_step_model = OneStepPolicy(state_dim, action_dim)

one_step_model, one_step_val_loss, one_step_mae = train_one_step(
    one_step_model
)

one_step_time = measure_time_one_step(one_step_model)

results.append({
    "model": "one_step_policy",
    "val_loss": one_step_val_loss,
    "raw_val_mae": one_step_mae,
    "actions_per_call": 1,
    "seconds_per_call": one_step_time,
    "seconds_per_action": one_step_time,
})


chunk_model = ActionChunkPolicy(
    state_dim=state_dim,
    action_dim=action_dim,
    chunk_size=CHUNK_SIZE
)

chunk_model, chunk_val_loss, chunk_mae, per_step_mae = train_chunk(
    chunk_model
)

chunk_time = measure_time_chunk(chunk_model)

results.append({
    "model": "action_chunk_policy",
    "val_loss": chunk_val_loss,
    "raw_val_mae": chunk_mae,
    "actions_per_call": CHUNK_SIZE,
    "seconds_per_call": chunk_time,
    "seconds_per_action": chunk_time / CHUNK_SIZE,
})


# ============================================================
# 13. PER-STEP CHUNK ERROR
# ============================================================

per_step_df = pd.DataFrame({
    "future_step": np.arange(CHUNK_SIZE),
    "raw_mae": per_step_mae,
})

per_step_df.to_csv("results/action_chunking_per_step_error.csv", index=False)

print("\nPer-step chunk error:")
print(per_step_df)


# ============================================================
# 14. SMOOTHNESS CHECK
# ============================================================

# Take one validation batch and compare expert vs predicted chunk smoothness.
chunk_model.eval()

example_batch = next(iter(val_loader))
example_state = example_batch["state"][:1].to(DEVICE)
expert_chunk = example_batch["action_chunk_raw"][0].numpy()

mean = torch.tensor(action_mean, dtype=torch.float32).to(DEVICE).view(1, 1, -1)
std = torch.tensor(action_std, dtype=torch.float32).to(DEVICE).view(1, 1, -1)

with torch.no_grad():
    pred_norm = chunk_model(example_state)
    pred_chunk = pred_norm * std + mean
    pred_chunk = pred_chunk.cpu().numpy()[0]

expert_smoothness = smoothness_score(expert_chunk)
pred_smoothness = smoothness_score(pred_chunk)

smoothness_df = pd.DataFrame([
    {
        "sequence": "expert_chunk",
        "smoothness_score": expert_smoothness,
    },
    {
        "sequence": "predicted_chunk",
        "smoothness_score": pred_smoothness,
    }
])

smoothness_df.to_csv("results/action_chunking_smoothness.csv", index=False)

print("\nSmoothness comparison:")
print(smoothness_df)


# ============================================================
# 15. SAVE FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df.to_csv("results/action_chunking_metrics.csv", index=False)

print("\nFinal action chunking comparison:")
print(results_df)

print("\nSaved results:")
print("results/action_chunking_metrics.csv")
print("results/action_chunking_per_step_error.csv")
print("results/action_chunking_smoothness.csv")

In [1]:
#hi